In [85]:
from functools import partial
from typing import Callable
import jax
import jax.numpy as jnp
from jax import Array
from jax.experimental import pallas as pl
from jax.experimental.pallas import triton as plgpu


def _kde_kernel_impl(
    X_train_ref, X_test_ref, out_ref, kernel_fn: Callable, BLOCK_M: int, BLOCK_N: int
):
    """
    Compute the kernel density estimate for a given block.
    This function is compiled and run in parallel on the device.
    """
    m = pl.program_id(axis=0)  # Index along test points
    n = pl.program_id(axis=1)  # Index along training points

    # Load the data for the current block
    x_test = X_test_ref[m]  # Single test point
    x_train = X_train_ref[n]  # Single train point

    # Apply kernel function
    val = kernel_fn(x_test, x_train)

    # Accumulate result in the output buffer
    out_ref[m, n] = val


def kde_kernel(
    X_train: Array,
    X_test: Array,
    kernel_fn: Callable,
    block_size: int = 128,
):
    """
    Kernel density estimation using a given kernel function.

    Parameters:
    - X_train: Training data points (N, D)
    - X_test: Test data points (M, D)
    - kernel_fn: Kernel function, must be JAX-compatible
    - block_size: Block size for parallel execution
    """
    M, D = X_test.shape
    N, _ = X_train.shape

    # Ensure the grid dimensions account for non-divisible block sizes
    grid_M = -(-M // block_size)  # Ceiling division for blocks in M
    grid_N = -(-N // block_size)  # Ceiling division for blocks in N

    # Define the shape of the output buffer
    out_shape = jax.ShapeDtypeStruct((M, N), dtype=jnp.float32)

    # Prepare the kernel function with bound parameters
    kernel = partial(_kde_kernel_impl, kernel_fn=kernel_fn, BLOCK_M=block_size, BLOCK_N=block_size)

    # Launch kernel computation using pallas_call
    output = pl.pallas_call(
        kernel,
        out_shape=out_shape,
        grid=(grid_M, grid_N),
        in_specs=[
            pl.BlockSpec((block_size, D),lambda i: (i * block_size,), ),  # X_train block
            pl.BlockSpec((block_size, D),lambda j: (j * block_size,)),  # X_test block
        ],
        out_specs=[
            pl.BlockSpec((block_size, block_size), lambda i, j: (i * block_size, j * block_size)),  # Output block
        ],
        interpret=True,
    )(X_train, X_test)

    # Sum along the training points axis to get the density estimate
    return jnp.sum(output, axis=1)


In [88]:
X_train = jax.random.normal(jax.random.PRNGKey(0), (1024, 2))
X_test = jax.random.normal(jax.random.PRNGKey(1), (128, 2))

def gaussian_kernel(x1, x2):
    print(x1.shape, x2.shape)
    return jnp.exp(-jnp.sum((x1 - x2) ** 2))

kde_kernel(X_train, X_test, kernel_fn= gaussian_kernel)


TypeError: kde_kernel.<locals>.<lambda>() takes 1 positional argument but 2 were given

In [129]:
import numpy as np
def matmul_kernel(x_ref, y_ref, z_ref):
  z_ref[...] = x_ref[...] @ y_ref[...]

def matmul(x: jax.Array, y: jax.Array, block_size: int =  16) -> jax.Array:
    grid_M = -(-x.shape[0] // block_size)
    grid_N = -(-y.shape[1] // block_size)

    return pl.pallas_call(
    matmul_kernel,
    out_shape=jax.ShapeDtypeStruct((x.shape[0], y.shape[1]), x.dtype),
    grid=(grid_M, grid_N),
    in_specs=[
        pl.BlockSpec((x.shape[0] // grid_M, x.shape[1]), lambda i, j: (i, 0)),
        pl.BlockSpec((y.shape[0], y.shape[1] // grid_N), lambda i, j: (0, j))
    ],
    out_specs=pl.BlockSpec(
        (x.shape[0] // grid_M, y.shape[1] // grid_N), lambda i, j: (i, j),
    )
  )(x, y)
k1, k2 = jax.random.split(jax.random.key(0))
x = jax.random.normal(k1, (512, 512))
y = jax.random.normal(k2, (512, 512))
z = matmul(x, y)

In [127]:
z - x @ y

Array([[ 0.002,  0.002,  0.011, ..., -0.032,  0.004, -0.004],
       [-0.046, -0.015, -0.05 , ..., -0.014,  0.007, -0.015],
       [ 0.006,  0.05 ,  0.002, ..., -0.021, -0.017,  0.017],
       ...,
       [-0.005,  0.013, -0.   , ..., -0.012, -0.011,  0.016],
       [ 0.014,  0.008,  0.004, ...,  0.011, -0.016, -0.006],
       [ 0.013,  0.012, -0.004, ..., -0.003, -0.028, -0.017]],      dtype=float32)